In [0]:
# ============================================================
# Bronze — Source 09: SFTP Drop
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=09_sftp/
# Target: bronze.sftp catalog in Unity Catalog
#
# Tables:
#   - bronze.sftp.supplier_catalog
#
# Raw format: CSV and Excel files (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "09_sftp"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=19/")

In [0]:
# Cell 2 — Read all CSV files
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path_csv = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/*.csv"

df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path_csv) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"CSV rows: {df_csv.count()}")
df_csv.printSchema()

In [0]:
# Cell 3 — Read all xlsx files explicitly
xlsx_files = []
for day in range(17, 25):
    try:
        files = dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day={day}/")
        for f in files:
            if f.name.endswith(".xlsx"):
                xlsx_files.append(f.path)
    except:
        pass

print(f"Found {len(xlsx_files)} xlsx files")

# Read each file and union
from functools import reduce
from pyspark.sql import DataFrame

dfs = []
for path in xlsx_files:
    df = spark.read.format("com.crealytics.spark.excel") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(path)
    dfs.append(df)

df_xlsx = reduce(DataFrame.union, dfs)
print(f"xlsx rows: {df_xlsx.count()}")
df_xlsx.printSchema()

In [0]:
# Cell 4 — Normalise schemas and union CSV + xlsx
from pyspark.sql.functions import col, lit

# Normalise CSV columns
df_csv_norm = df_csv.select(
    col("SKU").alias("product_sku"),
    col("`Product Name`").alias("product_name"),
    col("`Unit Cost GBP`").alias("unit_cost_gbp"),
    col("`Stock QTY`").alias("stock_qty"),
    col("`Lead Days`").alias("lead_days"),
    col("EAN").alias("ean"),
    lit(None).cast("string").alias("rrp_gbp"),
    lit(None).cast("string").alias("warehouse"),
    lit(None).cast("string").alias("updated_date"),
    lit("csv").alias("file_format")
)

# Normalise xlsx columns
df_xlsx_norm = df_xlsx.select(
    col("`Product Code`").alias("product_sku"),
    col("Description").alias("product_name"),
    col("`Cost Price GBP`").alias("unit_cost_gbp"),
    col("Stock").alias("stock_qty"),
    lit(None).cast("string").alias("lead_days"),
    lit(None).cast("string").alias("ean"),
    col("`RRP GBP`").alias("rrp_gbp"),
    col("Warehouse").alias("warehouse"),
    col("`Updated Date`").alias("updated_date"),
    lit("xlsx").alias("file_format")
)

# Union both
df_combined = df_csv_norm.union(df_xlsx_norm)
print(f"Combined rows: {df_combined.count()}")
df_combined.show(3)

In [0]:
# Cell 5 — Write to Bronze and update watermark
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.sftp")

df_combined.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.sftp.supplier_catalog")

# Get latest file timestamp across both CSV and xlsx
latest_ts_csv = df_csv.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
latest_ts = latest_ts_csv  # xlsx files don't expose _metadata via spark-excel

update_watermark(spark, SOURCE, latest_ts, df_combined.count())
print(f"✅ bronze.sftp.supplier_catalog: {df_combined.count()} rows (CSV + xlsx)")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.sftp.supplier_catalog").collect()[0]['cnt']
print(f"bronze.sftp.supplier_catalog: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '09_sftp'").show()